# Python JSON

> 📘 **Python Mastery** · Module 05 — Intermediate Python · Lesson 4/7

APIs, config files, databases and AI models all exchange data as JSON. Python's `json` module converts between that universal text format and your dictionaries in one call.

## 🎯 Learning Objectives

- Explain what JSON is and where you meet it
- Convert text <-> objects with `dumps` / `loads`
- Read and write `.json` files with `dump` / `load`
- Pretty-print with `indent` and `sort_keys`; compact with `separators`
- Navigate realistic nested API responses
- Work around JSON's limits (`datetime`, custom classes) using `default=str`

## 1. What Is JSON?

**JSON** (JavaScript Object Notation) is a plain-text format for structured data. Language-neutral -- Python, JavaScript, Java, Go and every database can read it -- it has become the default dialect of the web: REST APIs return it, config files use it, LLM tool-calls arrive in it.

Think of JSON as the international shipping box of data: whatever language built the contents, anyone can open the box.

```json
{
  "name": "Sarah",
  "age": 24,
  "skills": ["python", "sql"],
  "active": true,
  "manager": null
}
```

Rules to respect: keys in **double quotes**, no trailing commas, no comments, and only six kinds of value (object, array, string, number, `true`/`false`, `null`).

**Syntax:**
```python
import json
json.dumps(obj)   # Python object -> JSON TEXT (dump to 's'tring)
json.loads(text)  # JSON TEXT   -> Python object ('l'oad from 's'tring)
```

**Example:** a first round trip.

In [ ]:
import json

student = {"name": "Sarah", "age": 24, "courses": ["math", "cs"], "active": True}

text = json.dumps(student)          # object -> string
print(text)
print(type(text).__name__)

back = json.loads(text)             # string -> object
print(back["courses"], type(back).__name__)
print("round-trip equal?", back == student)

## 2. The Python <-> JSON Dictionary

The two worlds don't share types, so the module translates:

| Python | JSON | Note |
|---|---|---|
| `dict` | object | keys become strings |
| `list`, `tuple` | array | a `tuple` comes back as a `list` |
| `str` | string | |
| `int`, `float` | number | |
| `True` / `False` | `true` / `false` | capitalisation changes |
| `None` | `null` | |

Anything not in this table raises `TypeError` -- section 7 shows the escape hatch.

> 🔍 **Under the Hood:** `dumps` walks your object recursively and writes JSON text directly (the encoder is C-accelerated in CPython). Floats pass through `repr()`, which emits the *shortest* string that round-trips exactly -- which is why `json.dumps(0.1)` shows `0.1` rather than the full binary truth from Lesson 2. Dictionary keys must be strings or numbers; anything else is rejected before any output is produced.

**Syntax:**
```python
json.dumps({"tags": ("ml", "llm")})   # tuple -> ["ml", "llm"]
```

**Example:** watch tuples flatten into lists and `True` shrink to `true`.

In [ ]:
import json

data = {
    "id": 7,
    "tags": ("ml", "llm"),          # tuple going IN...
    "score": 91.5,
    "passed": True,
    "note": None,
}

text = json.dumps(data)
print(text)

back = json.loads(text)
print(back["tags"], type(back["tags"]).__name__)   # ...comes out a list

## 3. Reading and Writing JSON Files

Two more functions swap the string for an open file: `json.dump(obj, file)` writes, `json.load(file)` reads. Always open JSON files with `encoding="utf-8"` -- on Windows the default encoding can mangle non-English characters.

**Syntax:**
```python
with open(path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

with open(path, encoding="utf-8") as f:
    data = json.load(f)
```

**Example:** persist a cafe menu, then load it back.

In [ ]:
import json
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)
menu_path = Path("sample_data/menu.json")

menu = {
    "cafe": "Dhaka Beans",
    "items": [
        {"name": "espresso", "price_bdt": 220},
        {"name": "cold brew", "price_bdt": 280},
        {"name": "milk cake", "price_bdt": 190},
    ],
}

with open(menu_path, "w", encoding="utf-8") as f:
    json.dump(menu, f, indent=2)               # human-readable on disk

with open(menu_path, encoding="utf-8") as f:
    loaded = json.load(f)

print(loaded["items"][1])
print("total items:", len(loaded["items"]))

## 4. Pretty Printing: `indent`, `sort_keys`, `separators`

`dumps` by default emits one cramped line -- perfect for machines, miserable for humans. Keyword arguments fix readability (or squeeze it):

- `indent=2` -- one nesting level per two spaces;
- `sort_keys=True` -- alphabetical key order, so diffs between versions stay tidy;
- `separators=(",", ":")` -- the most compact legal output, ideal on the wire.

**Syntax:**
```python
json.dumps(data, indent=2, sort_keys=True)
json.dumps(data, separators=(",", ":"))
```

**Example:** same object, three sizes.

In [ ]:
import json

profile = {"city": "Chattogram", "age": 31, "name": "Arif"}

default_s = json.dumps(profile)
compact_s = json.dumps(profile, separators=(",", ":"))
pretty_s = json.dumps(profile, indent=2, sort_keys=True)

print(default_s, f"({len(default_s)} chars)")
print(compact_s, f"({len(compact_s)} chars)")
print(pretty_s)

## 5. Unicode and Non-ASCII Text

By default `json` escapes every non-ASCII character (`আ` becomes `\u0986`) so the output stays pure ASCII and safe anywhere. That's correct but unreadable. Pass `ensure_ascii=False` to keep real characters in the text -- then the surrounding file or stream must be UTF-8.

**Syntax:**
```python
json.dumps(data)                     # escapes to \uXXXX form
json.dumps(data, ensure_ascii=False) # keeps Bengali, accents, Chinese as-is
```

**Example:**

In [ ]:
import json

customer = {"name": "আরিফ হাসান", "city": "Dhaka"}

escaped = json.dumps(customer)
readable = json.dumps(customer, ensure_ascii=False)

print(escaped)     # portable, but who can read \u0986?
print(readable)    # humans win -- save this file as UTF-8

## 6. Navigating Nested JSON

Real API responses nest objects inside arrays inside objects. Navigation is just dictionary/list indexing chained together -- plus loops or comprehensions once you need every item.

**Syntax:**
```python
resp["data"]["users"][0]["orders"][0]["total"]
[o["total"] for u in users for o in u["orders"]]
```

**Example:** a miniature e-commerce response.

In [ ]:
import json

response = {
    "status": "ok",
    "rate_limit": {"remaining": 57},
    "data": {
        "users": [
            {"id": 1, "name": "Sarah", "city": "Dhaka",
             "orders": [{"id": "A100", "total": 1250.00}, {"id": "A101", "total": 480.50}]},
            {"id": 2, "name": "Arif", "city": "Chattogram",
             "orders": [{"id": "B200", "total": 3200.00}]},
        ]
    },
}

users = response["data"]["users"]
first_total = users[0]["orders"][1]["total"]
print("Sarah's second order:", first_total)

for u in users:
    print(f"{u['name']:<6} ({u['city']}): {len(u['orders'])} order(s)")

grand = sum(o["total"] for u in users for o in u["orders"])
big_spender = max(users, key=lambda u: sum(o["total"] for o in u["orders"]))
print(f"grand total {grand:.2f} BDT; top customer: {big_spender['name']}")

### Defensive navigation

Production responses break promises: fields vanish, lists come back empty. Reach safely with `.get()` and explicit checks instead of trusting hard indexes -- a missing key should give you a sensible default, not a crash mid-request.

**Syntax:**
```python
resp.get("data", {}).get("users", [])     # {} and [] instead of KeyError
```

**Example:**

In [ ]:
import json

text = '{"status": "ok", "data": {"users": []}}'   # nobody ordered today
resp = json.loads(text)

users = resp.get("data", {}).get("users", [])
print("user count:", len(users))

totals = [o["total"] for u in users for o in u.get("orders", [])]
print("grand total:", sum(totals))                 # 0 -- not a KeyError

## 7. When JSON Fails: Custom Objects

JSON knows nothing about *your* classes -- no `datetime`, no `Product`. Dumping one raises `TypeError`. The escape hatch is the `default=` hook: a function called for anything JSON can't handle, returning a JSON-safe stand-in.

- `default=str` -- quick, readable, one-way (you get strings back).
- `default=lambda o: o.__dict__` -- snapshots instance attributes into a dict.

Remember: after loading, everything is dicts/lists/strings again. JSON has no memory of your classes.

**Syntax:**
```python
json.dumps(obj, default=str)              # datetime -> "2026-08-26 ..."
json.dumps(obj, default=lambda o: o.__dict__)
```

**Example:**

In [ ]:
import json
from datetime import datetime

payload = {"user": "sarah", "logged_at": datetime(2026, 8, 26, 18, 45)}

try:
    json.dumps(payload)                       # boom
except TypeError as e:
    print("TypeError:", e)

fixed = json.dumps(payload, default=str)      # teach it a fallback
print(fixed)
back = json.loads(fixed)
print(type(back["logged_at"]).__name__, "->", back["logged_at"])   # a string now


class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price


text = json.dumps(Product("mechanical keyboard", 2500), default=lambda o: o.__dict__)
print(text)

## 8. `json` vs `pickle` in One Breath

Python also ships **pickle**, which serialises almost any Python object *to bytes* -- but only Python can read it, and unpickling untrusted data executes arbitrary code. For anything that leaves your machine: **use JSON**. Reserve pickle for trusted, Python-to-Python caching.

| | `json` | `pickle` |
|---|---|---|
| Output | text, universal | bytes, Python-only |
| Safe with untrusted input? | yes | **no** |
| Custom classes | needs `default=` | automatic |

**Pro tip:** for line-oriented datasets you'll meet **JSONL** -- one JSON object per line, parsed with one `json.loads(line)` each. Streaming-friendly and diff-friendly.

**Example:** write and read a tiny `.jsonl` event log.

In [ ]:
import json
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)
events = [
    {"event": "login", "user": "sarah"},
    {"event": "order", "user": "arif", "total": 480.5},
]

with open("sample_data/events.jsonl", "w", encoding="utf-8") as f:
    for e in events:
        f.write(json.dumps(e) + "\n")          # one object per line

with open("sample_data/events.jsonl", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        print(record["event"], "->", record.get("user"))

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Hand-writing JSON with single quotes or trailing commas | `json.JSONDecodeError` on load | double quotes everywhere; delete trailing commas |
| Opening files without `encoding="utf-8"` | mojibake for Bengali/accented names on Windows | always `open(path, encoding="utf-8")` |
| Expecting integer dict keys to survive | `{1: "a"}` comes back as `{"1": "a"}` -- silent change | convert keys yourself before dumping |
| Dumping a `set`, `datetime` or custom class | `TypeError: not JSON serializable` | convert (`sorted(s)`) or supply `default=` |
| Mixing up `loads`/`load` | `load` wants a file object, `loads` wants a string | remember the extra letter: s = string |
| Assuming tuples stay tuples | they round-trip into lists | accept lists, or convert explicitly |

## 💡 Best Practices & Pro Tips

- Write config/data files with `indent=2`; send them over the wire with `separators=(",", ":")` -- bytes cost money at scale.
- Treat external JSON as untrusted: check `status` fields and use `.get()` instead of hard indexing for keys that may be missing.
- For schemas you depend on, validate structure explicitly (`pydantic` models or `jsonschema`).
- Keep one canonical encoding -- UTF-8 -- across files, APIs and databases.
- **AI-engineering relevance:** LLM APIs are JSON end-to-end (messages, roles, tool calls). Models often emit JSON inside markdown fences -- strip the fence, then `json.loads`. Streaming datasets ship as JSONL: one `json.loads(line)` per line.

## 📌 Summary

| Tool | What it does | Example |
|---|---|---|
| `json.dumps(obj)` | object -> JSON string | `json.dumps({"a": 1})` |
| `json.loads(text)` | JSON string -> object | `json.loads('[1, 2]')` |
| `json.dump(o, f)` / `json.load(f)` | file versions of the above | `with open(p, encoding="utf-8") as f:` |
| `indent=2` | pretty-print | `json.dumps(d, indent=2)` |
| `sort_keys=True` | stable, diff-friendly key order | |
| `ensure_ascii=False` | keep real Unicode characters | Bengali names stay Bengali |
| `default=str` | survive custom objects | datetimes become strings |

Key takeaways:

- `dumps`/`loads` move strings, `dump`/`load` move files; the extra `s` means string.
- JSON's type table is small -- anything outside it needs conversion or a `default` hook.
- What loads back is always dicts, lists, strings and numbers -- never your original classes.
- UTF-8 everywhere, or Unicode will bite on Windows.

## 🔗 Next Lesson

Up next: **[05_Regex](../05_Regex/notes.ipynb)** -- pattern matching: find, extract and clean text like a surgeon.